<a href="https://colab.research.google.com/github/nikhitarao/nikhitarao_projects/blob/Healthcare-Insurance-Medical-Inflation-Analysis-using-Synthetic-Data/Inpatient_Healthcare_Medical_Procedure_Synthetic_Data_Generation_Policy_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Health Insurance Policy Database for a Fictional New York based Health Insurer

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

#### Define the starting constants



In [2]:
# Constants
start_year = 2010
end_year = 2023
base_year_policy_count = 50000
growth_rate = 0.05
fluctuation = 0.02

#### Define the Quarterly Distribution for New Business for seasonality

In [3]:
# Quarterly distribution for New Business
quarter_distribution = {
    "Q1": 0.30,
    "Q2": 0.20,
    "Q3": 0.15,
    "Q4": 0.35
}


#### Yearly Policy Counts Calculation with growth rate and fluctuations

In [4]:
# Function to calculate yearly policy counts with growth and fluctuation
def calculate_policy_count(base_count, years, growth_rate, fluctuation):
    yearly_counts = []
    for year in range(years):
        fluctuation_factor = np.random.uniform(-fluctuation, fluctuation)
        annual_growth = growth_rate + fluctuation_factor
        base_count = int(base_count * (1 + annual_growth))
        yearly_counts.append(base_count)
    return yearly_counts


In [5]:
# Generate yearly policy counts
years = end_year - start_year + 1
yearly_policy_counts = calculate_policy_count(base_year_policy_count, years, growth_rate, fluctuation)

#### Generating random dates with the Quarters

In [6]:
# Function to generate random dates within quarters
def generate_dates(year, quarter, count):
    quarter_ranges = {
        "Q1": (f"{year}-01-01", f"{year}-03-31"),
        "Q2": (f"{year}-04-01", f"{year}-06-30"),
        "Q3": (f"{year}-07-01", f"{year}-09-30"),
        "Q4": (f"{year}-10-01", f"{year}-12-31")
    }
    start_date, end_date = quarter_ranges[quarter]
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")
    return [start_date + timedelta(days=np.random.randint(0, (end_date - start_date).days + 1)) for _ in range(count)]


#### Data Type Definition for the Dataframe

In [7]:
# Define the data types for the DataFrame
column_dtypes = {
    "Business Type": "object",
    "Policy Start Date": "object",
    "Policy End Date": "object",
    "Churn Flag": "object",
    "Renewal Flag": "object",
    "Policyholder ID": "int64",
    "Location Type": "object",
    "Policy Type": "object",
    "Policy ID": "object",
    "Group Type": "object",
    "Annual Family Income": "float64",
    "Corporation Type": "object",
    "Industry Type": "object",
    "Annual Revenue": "float64",
    "Annual Payroll": "float64",
    "Number of Members": "int64",
    "Coverage Amount": "float64",
    "Policy Annual Premium": "float64"
}


#### Dataframe Creation with Business Type, Policy Start Date, Policy End Date populated

In [8]:
# Create an empty DataFrame with specified dtypes
dataset = pd.DataFrame(columns=column_dtypes.keys()).astype(column_dtypes)

In [9]:
# Placeholder for bulk data storage
data = []

policyholder_id_start = 10000000  # Start of 8-digit policyholder IDs
current_policyholder_id = policyholder_id_start  # Initialize the policyholder ID counter

for year, count in zip(range(start_year, end_year + 1), yearly_policy_counts):
    if year == 2010:
        # All policies in 2010 are "New Business"
        start_dates = [datetime(year, 1, 1) + timedelta(days=np.random.randint(0, 365)) for _ in range(count)]
        for start_date in start_dates:
            policy_end_date = start_date + timedelta(days=365 + (1 if start_date.year % 4 == 0 and (start_date.year % 100 != 0 or start_date.year % 400 == 0) else 0))

            data.append([
                "New Business", start_date.strftime("%Y-%m-%d"), policy_end_date.strftime("%Y-%m-%d"), None, None,
                current_policyholder_id, None, None, None, None, None, None, None, None, None, None, None, None
            ])
            current_policyholder_id += 1
    else:
        # Split into "New Business" and "Renewal" from 2011 onwards
        new_business_count = int(count * sum(quarter_distribution.values()))
        renewal_count = count - new_business_count

        # Distribute New Business across quarters
        quarterly_counts = {q: int(new_business_count * p) for q, p in quarter_distribution.items()}

        for quarter, quarter_count in quarterly_counts.items():
            start_dates = generate_dates(year, quarter, quarter_count)
            for start_date in start_dates:
                policy_end_date = start_date + timedelta(days=365 + (1 if start_date.year % 4 == 0 and (start_date.year % 100 != 0 or start_date.year % 400 == 0) else 0))

                data.append([
                    "New Business", start_date.strftime("%Y-%m-%d"), policy_end_date.strftime("%Y-%m-%d"), None, None,
                    current_policyholder_id, None, None, None, None, None, None, None, None, None, None, None, None
                ])
                current_policyholder_id += 1

        # Generate Renewal policies
        renewal_dates = [datetime(year, 1, 1) + timedelta(days=np.random.randint(0, 365)) for _ in range(renewal_count)]
        for renewal_date in renewal_dates:
            policy_end_date = renewal_date + timedelta(days=365 + (1 if renewal_date.year % 4 == 0 and (renewal_date.year % 100 != 0 or renewal_date.year % 400 == 0) else 0))

            data.append([
                "Renewal", renewal_date.strftime("%Y-%m-%d"), policy_end_date.strftime("%Y-%m-%d"), None, None,
                current_policyholder_id, None, None, None, None, None, None, None, None, None, None, None, None
            ])
            current_policyholder_id += 1

# Create DataFrame in one step
dataset = pd.DataFrame(data, columns=column_dtypes.keys())
print(f"Populated dataset with {len(dataset)} rows.")

dataset.head()

Populated dataset with 1060012 rows.


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,Annual Family Income,Corporation Type,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium
0,New Business,2010-03-27,2011-03-27,None,None,10000000,None,None,None,None,None,None,None,None,None,None,None,None
1,New Business,2010-10-07,2011-10-07,None,None,10000001,None,None,None,None,None,None,None,None,None,None,None,None
2,New Business,2010-09-20,2011-09-20,None,None,10000002,None,None,None,None,None,None,None,None,None,None,None,None
3,New Business,2010-10-04,2011-10-04,None,None,10000003,None,None,None,None,None,None,None,None,None,None,None,None
4,New Business,2010-04-04,2011-04-04,None,None,10000004,None,None,None,None,None,None,None,None,None,None,None,None


In [10]:
dataset.shape

(1060012, 18)

#### Adding values for columns Churn Flag, Renewal Flag, Policyholder ID


In [11]:
# Initialize policyholder map and churn rates
policyholder_map = {}  # Tracks Policyholder IDs and their status
churn_rate_range = (0.10, 0.20)  # Churn rate range
renewal_upsell_rate = 0.03  # Minimum upsell rate for renewals

# Add columns for processing
dataset["Churn Flag"] = None
dataset["Renewal Flag"] = None
dataset["Policyholder ID"] = None

# Track the last Policyholder ID
policyholder_id_counter = 1

# Process each year
for year in range(start_year, end_year + 1):
    # Extract data for the current year
    current_year_data = dataset[dataset["Policy Start Date"].str.startswith(str(year))].copy()

    # Separate "New Business" and "Renewal"
    new_business = current_year_data[current_year_data["Business Type"] == "New Business"]
    renewal = current_year_data[current_year_data["Business Type"] == "Renewal"]

    # Assign Policyholder IDs to new business
    new_business["Policyholder ID"] = np.arange(policyholder_id_counter, policyholder_id_counter + len(new_business))
    policyholder_id_counter += len(new_business)

    # Update policyholder map for new business
    for pid in new_business["Policyholder ID"]:
        policyholder_map[pid] = {"status": "Active", "year": year}

    # Process renewals
    if year > start_year:  # Renewals start from the second year
        previous_year = year - 1

        # Identify active IDs from the previous year
        active_ids = [
            pid for pid, details in policyholder_map.items()
            if details["status"] == "Active" and details["year"] == previous_year
        ]

        # Determine churn
        churn_rate = np.random.uniform(*churn_rate_range)
        churn_count = int(len(active_ids) * churn_rate)
        churned_ids = set(np.random.choice(active_ids, churn_count, replace=False))
        renewed_ids = set(active_ids) - churned_ids

        # Update policyholder map for churned and renewed policies
        for pid in churned_ids:
            policyholder_map[pid]["status"] = "Churned"
        for pid in renewed_ids:
            policyholder_map[pid]["year"] = year

        # Assign Renewal and Churn Flags
        renewal["Renewal Flag"] = renewal["Policyholder ID"].apply(
            lambda pid: "Renewed with Up-sell" if pid in renewed_ids and np.random.random() < renewal_upsell_rate else
                        ("Renewed As-is" if pid in renewed_ids else "Not Renewed")
        )
        renewal["Churn Flag"] = renewal["Policyholder ID"].apply(
            lambda pid: "Not Churn" if pid in renewed_ids else "Churn"
        )

    # Combine new business and processed renewals
    processed_data = pd.concat([new_business, renewal], ignore_index=True)

    # Ensure proper index alignment and update dataset
    dataset.loc[processed_data.index, ["Churn Flag", "Renewal Flag", "Policyholder ID"]] = processed_data[
        ["Churn Flag", "Renewal Flag", "Policyholder ID"]
    ]

print("Processing completed!")
dataset.head()

Processing completed!


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,Annual Family Income,Corporation Type,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium
0,New Business,2010-03-27,2011-03-27,None,None,955653,None,None,None,None,None,None,None,None,None,None,None,None
1,New Business,2010-10-07,2011-10-07,None,None,955654,None,None,None,None,None,None,None,None,None,None,None,None
2,New Business,2010-09-20,2011-09-20,None,None,955655,None,None,None,None,None,None,None,None,None,None,None,None
3,New Business,2010-10-04,2011-10-04,None,None,955656,None,None,None,None,None,None,None,None,None,None,None,None
4,New Business,2010-04-04,2011-04-04,None,None,955657,None,None,None,None,None,None,None,None,None,None,None,None


In [12]:
dataset.shape

(1060012, 18)

#### Adding values for columns Location Type, Policy Type, Policy ID

In [13]:
# Add columns for processing
dataset["Location Type"] = None
dataset["Policy Type"] = None
dataset["Policy ID"] = None

# Location distribution adjustments
base_distributions = {"Urban": 0.65, "Suburban": 0.25, "Rural": 0.10}
yearly_adjustments = {
    "Urban": lambda year: 0.65 - ((year - 2010) * 0.05 / 13),  # Linear decrease to 60%
    "Suburban": lambda year: 0.25 + ((year - 2010) * 0.05 / 13),  # Linear increase to 30%
    "Rural": lambda year: 0.10  # Stable around 10%
}

# Policy type definitions and distribution rules
policy_types = [
    {"Code": "I", "IntroYear": 2012, "Population": 0.30, "LocationDist": {"Urban": 0.50, "Suburban": 0.30, "Rural": 0.20}},
    {"Code": "F", "IntroYear": 2012, "Population": 0.25, "LocationDist": {"Urban": 0.40, "Suburban": 0.40, "Rural": 0.20}},
    {"Code": "G", "IntroYear": 2010, "Population": 0.30, "LocationDist": {"Urban": 0.60, "Suburban": 0.30, "Rural": 0.10}},
    {"Code": "M", "IntroYear": 2016, "Population": 0.10, "LocationDist": {"Urban": 0.30, "Suburban": 0.40, "Rural": 0.30}},
    {"Code": "MA", "IntroYear": 2014, "Population": 0.05, "LocationDist": {"Urban": 0.25, "Suburban": 0.50, "Rural": 0.25}},
    {"Code": "HD", "IntroYear": 2014, "Population": 0.10, "LocationDist": {"Urban": 0.55, "Suburban": 0.35, "Rural": 0.10}},
    {"Code": "MCP-HMO", "IntroYear": 2015, "Population": 0.25, "LocationDist": {"Urban": 0.50, "Suburban": 0.30, "Rural": 0.20}},
    {"Code": "MCP-PPO", "IntroYear": 2015, "Population": 0.25, "LocationDist": {"Urban": 0.45, "Suburban": 0.35, "Rural": 0.20}},
    {"Code": "SNP", "IntroYear": 2017, "Population": 0.01, "LocationDist": {"Urban": 0.30, "Suburban": 0.50, "Rural": 0.20}},
    {"Code": "SHI", "IntroYear": 2019, "Population": 0.01, "LocationDist": {"Urban": 0.60, "Suburban": 0.30, "Rural": 0.10}},
    {"Code": "CHI", "IntroYear": 2018, "Population": 0.01, "LocationDist": {"Urban": 0.70, "Suburban": 0.20, "Rural": 0.10}},
    {"Code": "DVI", "IntroYear": 2020, "Population": 0.03, "LocationDist": {"Urban": 0.40, "Suburban": 0.40, "Rural": 0.20}}
]

# Generate Policy ID
policy_id_counter = 1

# Process each year
for year in range(start_year, end_year + 1):
    # Calculate location distribution for the current year
    loc_dist = {loc: yearly_adjustments[loc](year) for loc in base_distributions}

    # Extract data for the current year
    current_year_data = dataset[dataset["Policy Start Date"].str.startswith(str(year))].copy()

    # Assign Location Type
    current_year_data["Location Type"] = np.random.choice(
        ["Urban", "Suburban", "Rural"], size=len(current_year_data), p=list(loc_dist.values())
    )

    # Assign Policy Type
    eligible_policy_types = [ptype for ptype in policy_types if ptype["IntroYear"] <= year]
    policy_type_weights = [ptype["Population"] for ptype in eligible_policy_types]

    # Normalize probabilities to sum to 1
    policy_type_weights = np.array(policy_type_weights) / sum(policy_type_weights)

    current_year_data["Policy Type"] = np.random.choice(
        [ptype["Code"] for ptype in eligible_policy_types], size=len(current_year_data), p=policy_type_weights
    )

    # Adjust Policy Type based on Location Distribution
    for loc in loc_dist:
        loc_indices = current_year_data[current_year_data["Location Type"] == loc].index
        loc_eligible_types = [
            ptype for ptype in eligible_policy_types if loc in ptype["LocationDist"]
        ]
        loc_weights = [ptype["LocationDist"][loc] for ptype in loc_eligible_types]

        # Normalize location weights
        loc_weights = np.array(loc_weights) / sum(loc_weights)

        current_year_data.loc[loc_indices, "Policy Type"] = np.random.choice(
            [ptype["Code"] for ptype in loc_eligible_types], size=len(loc_indices), p=loc_weights
        )

    # Assign Policy ID
    current_year_data["Policy ID"] = np.arange(policy_id_counter, policy_id_counter + len(current_year_data))
    policy_id_counter += len(current_year_data)

    # Update dataset
    dataset.loc[current_year_data.index, ["Location Type", "Policy Type", "Policy ID"]] = current_year_data[
        ["Location Type", "Policy Type", "Policy ID"]
    ]

print("Location Type, Policy Type, and Policy ID assignment completed!")

dataset.head()

Location Type, Policy Type, and Policy ID assignment completed!


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,Annual Family Income,Corporation Type,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium
0,New Business,2010-03-27,2011-03-27,None,None,955653,Urban,G,1,None,None,None,None,None,None,None,None,None
1,New Business,2010-10-07,2011-10-07,None,None,955654,Urban,G,2,None,None,None,None,None,None,None,None,None
2,New Business,2010-09-20,2011-09-20,None,None,955655,Urban,G,3,None,None,None,None,None,None,None,None,None
3,New Business,2010-10-04,2011-10-04,None,None,955656,Urban,G,4,None,None,None,None,None,None,None,None,None
4,New Business,2010-04-04,2011-04-04,None,None,955657,Suburban,G,5,None,None,None,None,None,None,None,None,None


In [14]:
dataset.shape

(1060012, 18)

#### Adding values for columns Group Type, Annual Family Income, Corporation Type, Industry Type, Annual Revenue, Annual Payroll

In [15]:
# Add new columns to the dataset
dataset["Group Type"] = None
dataset["Annual Family Income"] = None
dataset["Corporation Type"] = None
dataset["Industry Type"] = None
dataset["Annual Revenue"] = None
dataset["Annual Payroll"] = None

# Define distributions and value ranges
group_type_dist = {
    "Urban": [0.30, 0.70],
    "Suburban": [0.55, 0.45],
    "Rural": [0.75, 0.25],
}

family_income_ranges = {
    "Urban": (75000, 120000),
    "Suburban": (65000, 100000),
    "Rural": (50000, 80000),
}

corporation_type_dist = {
    "Urban": {"Large Corporation": 0.70, "Medium Corporation": 0.30},
    "Suburban": {"Medium Corporation": 1.00},
    "Rural": {"Small Corporation": 1.00},
}

industry_type_dist = {
    "Urban": {
        "Large Corporation": {"Tech": 0.25, "Finance": 0.45},
        "Medium Corporation": {"Healthcare": 0.10, "Corporate Services": 0.20},
    },
    "Suburban": {
        "Medium Corporation": {"Manufacturing": 0.25, "Retail": 0.35},
        "Small Corporation": {"Education": 0.15, "Local Businesses": 0.25},
    },
    "Rural": {
        "Small Corporation": {"Agriculture": 0.40, "Small Manufacturers": 0.60},
    },
}

revenue_ranges = {
    "Urban": {
        "Tech": (50000000, 100000000),
        "Finance": (50000000, 100000000),
        "Healthcare": (10000000, 50000000),
        "Corporate Services": (10000000, 50000000),
    },
    "Suburban": {
        "Manufacturing": (5000000, 30000000),
        "Retail": (5000000, 30000000),
        "Education": (1000000, 10000000),
        "Local Businesses": (1000000, 5000000),
    },
    "Rural": {
        "Agriculture": (1000000, 5000000),
        "Small Manufacturers": (5000000, 30000000),
    },
}

payroll_ranges = {
    "Urban": {
        "Tech": (10000000, 20000000),
        "Finance": (10000000, 20000000),
        "Healthcare": (5000000, 25000000),
        "Corporate Services": (5000000, 25000000),
    },
    "Suburban": {
        "Manufacturing": (2000000, 15000000),
        "Retail": (2000000, 15000000),
        "Education": (500000, 5000000),
        "Local Businesses": (500000, 5000000),
    },
    "Rural": {
        "Agriculture": (500000, 2000000),
        "Small Manufacturers": (2000000, 15000000),
    },
}

# Vectorized Group Type Assignment
location_types = dataset["Location Type"].values
group_types = np.empty(len(location_types), dtype=object)

for location, probs in group_type_dist.items():
    mask = location_types == location
    group_types[mask] = np.random.choice(["Family", "Group Health"], size=mask.sum(), p=probs)

dataset["Group Type"] = group_types

# Vectorized Income/Corporation/Industry Assignment
family_mask = dataset["Group Type"] == "Family"
group_health_mask = dataset["Group Type"] == "Group Health"

# Annual Family Income
for location, income_range in family_income_ranges.items():
    mask = family_mask & (dataset["Location Type"] == location)
    dataset.loc[mask, "Annual Family Income"] = np.random.randint(income_range[0], income_range[1] + 1, size=mask.sum())

# Corporation Type for Group Health
for location, corp_probs in corporation_type_dist.items():
    mask = group_health_mask & (dataset["Location Type"] == location)
    corp_types = np.random.choice(
        list(corp_probs.keys()), size=mask.sum(), p=list(corp_probs.values())
    )
    dataset.loc[mask, "Corporation Type"] = corp_types

# Industry Type and Financials
for location, industry_map in industry_type_dist.items():
    corp_types = dataset.loc[group_health_mask & (dataset["Location Type"] == location), "Corporation Type"].unique()
    for corp_type in corp_types:
        industry_probs = industry_map.get(corp_type, {})
        industry_probs_norm = np.array(list(industry_probs.values())) / sum(industry_probs.values())

        # Filter rows matching the location and corporation type
        mask = (dataset["Corporation Type"] == corp_type) & (dataset["Location Type"] == location)
        industries = np.random.choice(
            list(industry_probs.keys()), size=mask.sum(), p=industry_probs_norm
        )

        # Assign Industry Type
        dataset.loc[mask, "Industry Type"] = industries

        # Assign Annual Revenue and Payroll in chunks to ensure size matches
        for industry in list(industry_probs.keys()):
            industry_mask = mask & (dataset["Industry Type"] == industry)
            industry_size = industry_mask.sum()

            # Assign Annual Revenue
            revenue_range = revenue_ranges[location].get(industry)
            if revenue_range:
                dataset.loc[industry_mask, "Annual Revenue"] = np.random.randint(
                    revenue_range[0], revenue_range[1] + 1, size=industry_size
                )

            # Assign Annual Payroll
            payroll_range = payroll_ranges[location].get(industry)
            if payroll_range:
                dataset.loc[industry_mask, "Annual Payroll"] = np.random.randint(
                    payroll_range[0], payroll_range[1] + 1, size=industry_size
                )


print("Optimized population of Group Type, Income, Corporation, Industry, Revenue, and Payroll completed!")
dataset.head()

Optimized population of Group Type, Income, Corporation, Industry, Revenue, and Payroll completed!


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,Annual Family Income,Corporation Type,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium
0,New Business,2010-03-27,2011-03-27,None,None,955653,Urban,G,1,Family,84054,None,None,None,None,None,None,None
1,New Business,2010-10-07,2011-10-07,None,None,955654,Urban,G,2,Group Health,None,Medium Corporation,Healthcare,35233426,7862566,None,None,None
2,New Business,2010-09-20,2011-09-20,None,None,955655,Urban,G,3,Group Health,None,Large Corporation,Tech,86307291,16536402,None,None,None
3,New Business,2010-10-04,2011-10-04,None,None,955656,Urban,G,4,Group Health,None,Large Corporation,Finance,57302960,10329155,None,None,None
4,New Business,2010-04-04,2011-04-04,None,None,955657,Suburban,G,5,Group Health,None,Medium Corporation,Manufacturing,12314594,14685886,None,None,None


In [16]:
dataset.shape

(1060012, 18)

#### Adding values for columns Number of Members. Coverage Amount and Policy Annual Premium


In [17]:
# Add new columns for processing
dataset["Number of Members"] = None
dataset["Coverage Amount"] = None
dataset["Policy Annual Premium"] = None

# Define member distributions for Group Type
members_dist_family = {
    "Urban": {"range": (3, 5), "weight": 0.5},
    "Suburban": {"range": (4, 6), "weight": 0.5},
    "Rural": {"range": (5, 8), "weight": 1.0},
}

members_dist_group = {
    "Urban": {"Tech and Finance": (200, 1000, 0.7), "Healthcare and Corporate Services": (100, 500, 0.3)},
    "Suburban": {"Manufacturing and Retail": (50, 300, 0.6), "Education": (30, 200, 0.4)},
    "Rural": {"Agriculture and Small Manufacturers": (25, 200, 1.0)},
}

# Coverage amount ranges for Policy Type
coverage_ranges = {
    "I": (10000, 50000),
    "F": (50000, 200000),
    "G": (500000, 5000000),
    "M": (20000, 75000),
    "MA": (50000, 100000),
    "HD": (5000, 20000),
    "SHI": (1000, 10000),
    "DVI": (1000, 5000),
    "SNP": (50000, 75000),
    "CHI": (1000, 10000),
}

# Premium ranges for Policy Type
premium_ranges = {
    "I": (500, 2000),
    "F": (2000, 10000),
    "G": (50000, 500000),
    "M": (1000, 5000),
    "MA": (1000, 5000),
    "HD": (400, 1400),
    "SHI": (500, 1000),
    "DVI": (100, 500),
    "SNP": (2000, 5000),
    "CHI": (500, 2000),
}

# Vectorized processing
location_types = dataset["Location Type"].values
group_types = dataset["Group Type"].values
policy_types = dataset["Policy Type"].values

# Populate Number of Members
for loc in members_dist_family.keys():
    family_mask = (location_types == loc) & (group_types == "Family")
    group_mask = (location_types == loc) & (group_types == "Group Health")

    # Family Members
    family_range = members_dist_family[loc]["range"]
    dataset.loc[family_mask, "Number of Members"] = np.random.randint(family_range[0], family_range[1] + 1, family_mask.sum())

    # Group Health Members
    for category, (low, high, weight) in members_dist_group[loc].items():
        category_mask = group_mask & dataset["Industry Type"].str.contains(category)
        dataset.loc[category_mask, "Number of Members"] = np.random.randint(low, high + 1, category_mask.sum())

# Populate Coverage Amount
for policy, (low, high) in coverage_ranges.items():
    mask = policy_types == policy
    dataset.loc[mask, "Coverage Amount"] = np.random.randint(low, high + 1, mask.sum())

# Add adjustments for Location and Group Type
dataset.loc[location_types == "Urban", "Coverage Amount"] *= 1.1
dataset.loc[location_types == "Suburban", "Coverage Amount"] *= 1.05

# Populate Policy Annual Premium
for policy, (low, high) in premium_ranges.items():
    mask = policy_types == policy
    dataset.loc[mask, "Policy Annual Premium"] = np.random.randint(low, high + 1, mask.sum())

# Adjust premiums for location
dataset.loc[location_types == "Urban", "Policy Annual Premium"] *= 1.2
dataset.loc[location_types == "Suburban", "Policy Annual Premium"] *= 1.1

dataset.head()

,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,Annual Family Income,Corporation Type,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium
0,New Business,2010-03-27,2011-03-27,None,None,955653,Urban,G,1,Family,84054,None,None,None,None,4,1958931.7,338937.6
1,New Business,2010-10-07,2011-10-07,None,None,955654,Urban,G,2,Group Health,None,Medium Corporation,Healthcare,35233426,7862566,None,4568526.6,318918.0
2,New Business,2010-09-20,2011-09-20,None,None,955655,Urban,G,3,Group Health,None,Large Corporation,Tech,86307291,16536402,None,3273598.9,326126.4
3,New Business,2010-10-04,2011-10-04,None,None,955656,Urban,G,4,Group Health,None,Large Corporation,Finance,57302960,10329155,None,4078770.3,297261.6
4,New Business,2010-04-04,2011-04-04,None,None,955657,Suburban,G,5,Group Health,None,Medium Corporation,Manufacturing,12314594,14685886,None,5025936.3,345757.5


In [18]:
dataset.shape

(1060012, 18)

#### Writing the Synthetic Policy Dataset to a CSV File and then Downloaded


In [22]:
# Save the dataset to a local file in Colab
file_path = 'inpatient_healthcare_insurance_synthetic_policy_dataset.csv'
dataset.to_csv(file_path, index=False)
print(f"Dataset saved temporarily at {file_path}")

Dataset saved temporarily at inpatient_healthcare_insurance_synthetic_policy_dataset.csv


In [23]:
from google.colab import files

# Download the file
files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>